# 05 — Size scaling

**Sweep 2.** Vary `n in {4, 6, 8, 10, 12, 14, 16}` by drawing 20 random
size-n subsets of the canonical 16-equity universe (single canonical
subset for n=16). For each subset we recompute the QUBO from the
sub-(μ, Σ) and run every solver, then report median + IQR across
subsets. The `n=16` endpoint by construction matches the run in
notebooks 02/03/04/06.

`K = round(0.25 * n)` so at n=16 we recover K=4 = `DEFAULTS["K_AT_16"]`.

Honest expectation at these sizes: classical methods are strictly better
— faster, more reliable, no barren plateaus. QAOA matches optimality
but doesn't outperform. The methodology is the point.

In [ ]:
# === Bootstrap (Colab + local) ===
import os, urllib.request as _u
exec((open('../scripts/bootstrap.py') if os.path.exists('../scripts/bootstrap.py') else _u.urlopen('https://raw.githubusercontent.com/egil10/fys5419/main/project2/code/scripts/bootstrap.py')).read())

# === Project imports ===
import json
import numpy as np
import pandas as pd

from scripts.colab     import out_dir
from scripts.data      import load_universe
from scripts.portfolio import PortfolioProblem, DEFAULTS
from scripts.classical import brute_force, greedy_top_k, simulated_annealing, markowitz_round
from scripts.qaoa      import solve
from scripts.metrics   import prob_optimal

RESULTS = out_dir('results')
print(f'Results will be saved to: {RESULTS}')

In [ ]:
# === Scaling protocol ===
# Use the canonical 16-asset universe and SUB-SAMPLE for each n.
# For each n in N_VALUES, draw M random size-n subsets (one canonical
# subset for n=16) and run every solver on the (mu, Sigma) restricted
# to that subset. Report median + IQR across subsets.
#
# K is the ratio rule (K = round(K_FRAC * n)) — keeps problem difficulty
# roughly constant. K_AT_16 = round(0.25 * 16) = 4, so the n=16 endpoint
# of this sweep matches notebooks 02/03/04/06 exactly.

N_VALUES   = [4, 6, 8, 10, 12, 14, 16]
M_SUBSETS  = 20                 # random subsets per n; n=16 is single canonical
P          = 3                  # QAOA depth
N_RESTARTS = 10                 # multi-start seeds for QAOA + SA
LAM        = DEFAULTS['lam']
AP         = DEFAULTS['A']
K_FRAC     = DEFAULTS['K_FRAC']
SEED       = 42

universe = load_universe()
print(f'universe: n={universe.n}  tickers={universe.tickers}')

In [ ]:
cache = RESULTS / 'size_scaling.json'

def _solve_subset(sub, K):
    """Run every solver on a sub-universe and return one record per solver."""
    pf = PortfolioProblem(sub.mu, sub.Sigma, lam=LAM, A=AP, K=K,
                          tickers=sub.tickers)
    bf = brute_force(pf)
    gr = greedy_top_k(pf, score='sharpe')
    mr = markowitz_round(pf)

    sa_runs = [simulated_annealing(pf, n_sweeps=1000*pf.n, seed=s)
               for s in range(N_RESTARTS)]
    sa_costs = np.array([sr.cost for sr in sa_runs])
    sa_best  = float(sa_costs.min())
    sa_time  = float(np.median([sr.runtime for sr in sa_runs]))

    qres = solve(pf, p=P, n_restarts=N_RESTARTS, seed=SEED)

    return [
        {'solver': 'brute_force',     'cost': bf.cost,      'runtime_s': bf.runtime,  'ratio': 1.0},
        {'solver': 'greedy_sharpe',   'cost': gr.cost,      'runtime_s': gr.runtime,  'ratio': gr.cost / bf.cost},
        {'solver': 'markowitz_round', 'cost': mr.cost,      'runtime_s': mr.runtime,  'ratio': mr.cost / bf.cost},
        {'solver': 'sa_best',         'cost': sa_best,      'runtime_s': sa_time,     'ratio': sa_best / bf.cost},
        {'solver': f'qaoa_p{P}',      'cost': float(qres['energy']),
         'runtime_s': qres['runtime'], 'ratio': float(qres['energy']) / bf.cost,
         'p_optimal': prob_optimal(qres['probs'], bf.x)},
    ]


if cache.exists():
    rows = json.loads(cache.read_text())
    print(f'loaded {cache.name}  ({len(rows)} records)')
else:
    rng  = np.random.default_rng(SEED)
    rows = []
    for n in N_VALUES:
        K = max(1, round(K_FRAC * n))

        # Subsets: nested for n=16 (the canonical full universe is the only
        # one). For smaller n, draw M_SUBSETS random subsets without replacement.
        if n == universe.n:
            subset_idxs = [tuple(range(universe.n))]
        else:
            subset_idxs = []
            for _ in range(M_SUBSETS):
                idx = tuple(sorted(rng.choice(universe.n, size=n, replace=False)))
                subset_idxs.append(idx)

        for subset_id, idx in enumerate(subset_idxs):
            sub = universe.subset(idx)
            for rec in _solve_subset(sub, K):
                rec.update({'n': n, 'K': K, 'subset_id': subset_id,
                            'subset_idx': list(idx)})
                rows.append(rec)
        print(f'  n={n:2d}  K={K}  subsets={len(subset_idxs)}')

    cache.write_text(json.dumps(rows, indent=2))
    print(f'saved -> {cache.name}  ({len(rows)} records)')

df = pd.DataFrame(rows)
df.groupby(['n', 'solver'])['ratio'].agg(['median', 'mean', 'std', 'count'])